In [42]:
# CREATE TABLE fact_lap (
#     lap_id bigserial primary key,
#     session_id int not null,
#     driver_id int not null,
#     compound_id int,
#     lap_number int,
#     lap_time_s numeric,
#     sector1_s numeric,
#     sector2_s numeric,
#     sector3_s numeric,
#     tyre_life int,
#     stint_number int,
#     fuel_load_est numeric,
#     position int,
#     gap_to_leader_s numeric,
#     track_status varchar(5),
#     is_pit_in_lap boolean,
#     is_pit_out_lap boolean,
#     is_personal_best boolean,
#     is_accurate boolean,

#     CONSTRAINT fk_lap_session  FOREIGN KEY (session_id)  REFERENCES dim_session(session_id),
#     CONSTRAINT fk_lap_driver   FOREIGN KEY (driver_id)   REFERENCES dim_driver(driver_id),
#     CONSTRAINT fk_lap_compound FOREIGN KEY (compound_id) REFERENCES dim_compound(compound_id)
# );
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import pandas as pd
import fastf1

fastf1.Cache.enable_cache('/tmp/fastf1_cache')

spark = SparkSession.builder \
        .appName("F1_Telemetry_Processing") \
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
        .getOrCreate()

url = "jdbc:postgresql://localhost:5432/telemetry"
properties = {"user": "michal", "password": "root", "driver": "org.postgresql.Driver"}

In [3]:
dim_season = spark.read.jdbc(url=url, table='dim_season', properties=properties)
dim_season_df = dim_season.collect()
dim_session = spark.read.jdbc(url=url, table='dim_session', properties=properties)
dim_session_df = dim_session.collect()
dim_driver = spark.read.jdbc(url=url, table='dim_driver', properties=properties)
dim_driver_df = dim_driver.collect()
dim_event = spark.read.jdbc(url=url, table='dim_event', properties=properties)
dim_event_df = dim_event.collect()
dim_compound = spark.read.jdbc(url=url, table='dim_compound', properties=properties)
dim_compound_df = dim_compound.collect()

season_map = {row['season_id']: row['year'] for row in dim_season_df}
driver_map = {row['code']: row['driver_id'] for row in dim_driver_df}
event_map = {
    row["event_id"]: (row["season_id"], row["round_number"])
    for row in dim_event.collect()
}
compound_map = {row['name']: row['compound_id'] for row in dim_compound_df}

In [67]:
def build_fact_lap(session, session_id, driver_map, compound_map, spark):
    laps = session.laps.copy()

    if laps.empty:
        print(f"Brak lapów dla session_id={session_id}")
        return None

    laps["is_pit_in_lap"]  = laps["PitInTime"].notna()
    laps["is_pit_out_lap"] = laps["PitOutTime"].notna()

    # gap_to_leader
    laps["lap_end_session_time"] = laps["Sector3SessionTime"].dt.total_seconds()

    # czas lidera per okrążenie
    leader_times = laps.groupby("LapNumber")["lap_end_session_time"].min().reset_index()
    leader_times.columns = ["LapNumber", "leader_session_time"]
    
    laps = laps.merge(leader_times, on="LapNumber", how="left")
    laps["gap_to_leader_s"] = laps["lap_end_session_time"] - laps["leader_session_time"]

    # ~110kg na start, spalanie ~1.8kg/lap
    laps["fuel_load_est"] = (110 - laps["LapNumber"] * 1.8).clip(lower=0)
    laps = laps[["Driver", "Compound", 
                 "LapNumber", "LapTime", "Sector1Time", "Sector2Time", "Sector3Time",
                 "TyreLife", "Stint", "fuel_load_est",
                 "Position", "gap_to_leader_s", "TrackStatus",
                 "is_pit_in_lap", "is_pit_out_lap",
                 "IsPersonalBest", "IsAccurate"]]

    # timedelta → sekundy
    for col in laps.columns:
        if pd.api.types.is_timedelta64_dtype(laps[col]):
            laps[col] = laps[col].dt.total_seconds()

    laps["IsPersonalBest"] = laps["IsPersonalBest"].fillna(False).astype(bool)
    
    laps_df = spark.createDataFrame(laps)


    laps_df = laps_df.withColumn("session_id", F.lit(session_id))
    
    driver_mapping   = F.create_map([F.lit(x) for pair in driver_map.items()   for x in pair])
    compound_mapping = F.create_map([F.lit(x) for pair in compound_map.items() for x in pair])

    laps_df = laps_df.withColumn("driver_id",   driver_mapping[F.col("Driver")])
    laps_df = laps_df.withColumn("compound_id", compound_mapping[F.col("Compound")])

    laps_df = laps_df.filter(F.col("driver_id").isNotNull())
    
    laps_df = laps_df.withColumnsRenamed({
        "LapNumber": "lap_number",
        "LapTime": "lap_time_s",
        "Sector1Time": "sector1_s",
        "Sector2Time": "sector2_s",
        "Sector3Time": "sector3_s",
        "TyreLife": "tyre_life",
        "Stint": "stint_number",
        "Position": "position",
        "TrackStatus": "track_status",
        "IsPersonalBest": "is_personal_best",
        "IsAccurate": "is_accurate",
    })
    
    laps_df = laps_df.select(["session_id", "driver_id", "compound_id", 
                 "lap_number", "lap_time_s", "sector1_s", "sector2_s", "sector3_s",
                 "tyre_life", "stint_number", "fuel_load_est",
                 "position", "gap_to_leader_s", "track_status",
                 "is_pit_in_lap", "is_pit_out_lap",
                 "is_personal_best", "is_accurate"
                ])

    # NaN -> null dla float
    float_cols = [f.name for f in laps_df.schema.fields
                  if isinstance(f.dataType, (DoubleType, FloatType))]
    for col in float_cols:
        laps_df = laps_df.withColumn(col,
            F.when(F.isnan(F.col(col)), None).otherwise(F.col(col)))

    # pusty string -> null
    str_cols = [f.name for f in laps_df.schema.fields
                if str(f.dataType) == "StringType()"]
    for col in str_cols:
        laps_df = laps_df.withColumn(col,
            F.when(F.col(col) == "", None).otherwise(F.col(col)))

    laps_df = laps_df \
        .withColumn("lap_number",   F.expr("try_cast(lap_number   as INT)")) \
        .withColumn("tyre_life",    F.expr("try_cast(tyre_life    as INT)")) \
        .withColumn("stint_number", F.expr("try_cast(stint_number as INT)")) \
        .withColumn("position",     F.expr("try_cast(position     as INT)")) \
        .withColumn("driver_id",    F.expr("try_cast(driver_id    as INT)")) \
        .withColumn("compound_id",  F.expr("try_cast(compound_id  as INT)"))

    return laps_df

In [68]:
loaded = 0
errors = 0
for row in dim_session_df:
    season_id, round_number = event_map[row.event_id]

    try:
        session = fastf1.get_session(season_map[season_id], round_number, row.session_type)
        session.load(laps=True, telemetry=False, weather=False, messages=False)
    except Exception as e:
        print(f"Failed: {season_map[season_id]} R{round_number} {row.session_type}: {e}")
        errors += 1
        continue
    # print(session.laps.head())
    sdf = build_fact_lap(session, row.session_id, driver_map, compound_map, spark)
    sdf.write.jdbc(url=url, table='fact_lap', mode='append', properties=properties)
    loaded += 1
print(f'loaded {loaded/(loaded+errors)*100}% ({loaded}, {errors})')

core           INFO 	Loading data for Austrian Grand Prix - Practice 1 [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Austrian Grand Prix - Practice 1 [v3.8.1]
req            INFO 	Using cached data for session_info
INFO:fastf1.fastf1.req:Using cached data for session_info
req            INFO 	Using cached data for driver_info
INFO:fastf1.fastf1.req:Using cached data for driver_info
DEBUG:fastf1.ergast:Failed to parse timestamp '' in Ergastresponse.
req            INFO 	Using cached data for session_status_data
INFO:fastf1.fastf1.req:Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
INFO:fastf1.fastf1.req:Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
INFO:fastf1.fastf1.req:Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
INFO:fastf1.fastf1.req:Using cached data for timing_app_data
core           INFO 	Processing timing 

Failed: 2019 R7 FP2: any API: 500 calls/h


core           INFO 	Loading data for Canadian Grand Prix - Practice 3 [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Canadian Grand Prix - Practice 3 [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


Failed: 2025 R10 FP3: any API: 500 calls/h


core           INFO 	Loading data for Canadian Grand Prix - Qualifying [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Canadian Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


...

Failed: 2024 R9 FP3: any API: 500 calls/h


Traceback (most recent call last):
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
core           INFO 	Loading data for Canadian Grand Prix - Qualifying [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Canadian Grand Prix - Qualifying [v3.8.1]
req            INFO 	No cached data found for session_info. Load

Failed: 2024 R9 Q: any API: 500 calls/h


Traceback (most recent call last):
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 311, in _resend
    response = self._send_and_cache(request, actions, cached_response, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/requests_cache/session.py", line 279, in _send_and_cache
    response = super().send(request, **kwargs)
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 110, in send
    lim.limit()
  File "/home/michal/miniconda3/envs/pyspark_env/lib/python3.10/site-packages/fastf1/req.py", line 80, in limit
    raise RateLimitExceededError(self._info)
fastf1.exceptions.RateLimitExceededError: any API: 500 calls/h
core           INFO 	Loading data for Canadian Grand Prix - Race [v3.8.1]
INFO:fastf1.fastf1.core:Loading data for Canadian Grand Prix - Race [v3.8.1]
req            INFO 	No cached data found for session_info. Loading data...


Failed: 2024 R9 R: any API: 500 calls/h
Failed: 2022 R15 FP1: any API: 500 calls/h
Failed: 2022 R15 FP2: any API: 500 calls/h
Failed: 2022 R15 FP3: any API: 500 calls/h
Failed: 2022 R15 Q: any API: 500 calls/h
Failed: 2022 R15 R: any API: 500 calls/h
Failed: 2022 R16 FP1: any API: 500 calls/h
Failed: 2022 R16 FP2: any API: 500 calls/h
Failed: 2022 R16 FP3: any API: 500 calls/h
Failed: 2022 R16 Q: any API: 500 calls/h
Failed: 2022 R16 R: any API: 500 calls/h
